# SILVER → GOLD: Agregacion de Customers

Este notebook ejecuta el script de agregacion que:
1. Lee la capa Silver (Parquet)
2. Agrupa por estado y calcula metricas de negocio
3. Guarda el resumen en la capa Gold (Parquet)

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

## 1. Verificar que la capa Silver existe

In [ ]:
SILVER_PATH = PROJECT_ROOT / 'data' / 'silver' / 'customers'

if not SILVER_PATH.exists():
    raise FileNotFoundError(
        f'Silver layer no encontrada en {SILVER_PATH}.\n'
        'Ejecuta primero el notebook 02_bronze_to_silver_customers.ipynb'
    )
print(f'Silver layer encontrada en: {SILVER_PATH}')

## 2. Ejecutar el script SILVER → GOLD

In [ ]:
from src.silver_to_gold.customers_aggregation import aggregate_customers

total_rows = aggregate_customers()
print(f'\nAgregacion completada: {total_rows} filas en Gold.')

## 3. Explorar la capa Gold

In [ ]:
from src.utils.spark_session import create_spark_session

GOLD_PATH = PROJECT_ROOT / 'data' / 'gold' / 'customers_summary'

spark = create_spark_session('notebook_gold_validation')
df_gold = spark.read.parquet(str(GOLD_PATH))

print('Schema Gold:')
df_gold.printSchema()

In [ ]:
print('Resumen de customers por estado:')
df_gold.orderBy('total_customers', ascending=False).show(truncate=False)

In [ ]:
# Visualizacion con pandas
df_pandas = df_gold.toPandas()
df_pandas.sort_values('total_customers', ascending=False)

In [ ]:
spark.stop()
print('SparkSession cerrada.')